# 简单组合回测

In [ ]:
import empyrical as ep
import pandas as pd
from datetime import datetime,timedelta

In [ ]:
df=get_all_securities(['fund'])
df[df.display_name.str.contains("纳斯达克")]

In [ ]:
codes = [
    ('513300.XSHG', '纳斯达克ETF',0.3),
    ('512890.XSHG', '红利低波',0.4),
    ('518880.XSHG', '黄金ETF',0.3),
]

## 成分股走势

In [ ]:
df_list=[]
for code,name,weight in codes:    
    df = get_price(
        code, 
        end_date=datetime.now(),
        count=3*365,
        frequency='1d', 
        fields=['close'], 
        skip_paused=False, 
        fq='pre', 
        panel=False, 
        fill_paused=True)
    df = df.rename(columns={'close': name})
    df_list.append(df)
all_df = pd.concat(df_list, axis=1).ffill()

In [ ]:
all_df.plot(figsize=(10,5))

## 每日再平衡回测

In [ ]:
# 假设：每日再平衡，该算法隐含了"每天收盘按权重调仓"的假设。
# 1. 算每个资产的日收益率
returns_dict = {}
for code,name,weight in codes:
    returns_dict[name] = all_df[name].pct_change()
# 2. 拼成一个 df（自动对齐日期，缺失会变 NaN）
returns = pd.concat(returns_dict, axis=1).ffill().dropna(how='any')
# 3. 权重（按比例归一化，3:4:3）
weights = pd.Series([x[2] for x in codes], index=returns.columns)
# 4. 组合日收益率 = 加权平均
port_ret = (returns * weights).sum(axis=1)
print("年化收益:", ep.annual_return(port_ret))
print("年化波动:", ep.annual_volatility(port_ret))
print("Sharpe:", ep.sharpe_ratio(port_ret))
print("Sortino:", ep.sortino_ratio(port_ret))
print("最大回撤:", ep.max_drawdown(port_ret))
port_ret_df = pd.DataFrame({'cumulative': (1 + port_ret).cumprod() - 1})
port_ret_df.plot(figsize=(10,5))

## buy & hold 不调仓回测

In [ ]:
# 如果是不再平衡（buy & hold，按初始资金配比后持有），那要换算法：
# 先用初始价格算出资金占比，再算每只资产的份额
init_prices = all_df.iloc[0]
shares = weights / init_prices        # 每只资产买多少"份"
holdings = df * shares                  # 每天持仓市值
nav = holdings.sum(axis=1)              # 总净值
bh_ret = nav.pct_change()               # buy & hold 日收益
print("年化收益:", ep.annual_return(bh_ret))
print("年化波动:", ep.annual_volatility(bh_ret))
print("Sharpe:", ep.sharpe_ratio(bh_ret))
print("Sortino:", ep.sortino_ratio(bh_ret))
print("最大回撤:", ep.max_drawdown(bh_ret))
bh_ret_df = pd.DataFrame({'cumulative': (1 + bh_ret).cumprod() - 1})
bh_ret_df.plot(figsize=(10,5))

## 月度调仓

**月度再平衡 + 月内权重漂移**
- 调仓日把权重拉回目标
- 调仓日到下次调仓日之间，权重随价格自然漂移
- 这才是实战里真实的"季度/月度再平衡"

In [ ]:
def monthly_rebalance_portfolio(prices: pd.DataFrame, 
                                 weights, 
                                 freq: str = 'ME') -> pd.Series:
    """
    prices  : DataFrame, 列是资产, 索引是日期
    weights : dict 或 Series, 如 {'hs300': 0.3, 'dividend': 0.3, 'gold': 0.4}
    freq    : 调仓频率, 'M' 月末 / 'Q' 季末 / 'Y' 年末 / 'W' 周
    return  : 组合日收益率 Series
    """
    # 1. 权重归一化
    w = pd.Series(weights, index=prices.columns, dtype=float)
    w = w / w.sum()

    # 2. 日收益率
    daily_ret = prices.pct_change().dropna()

    # 3. 调仓日 = 每月最后交易日
    rebalance_dates = set(daily_ret.resample(freq).last().index)

    # 4. 模拟：每天根据"今日开盘时的权重"算收益
    port_ret = pd.Series(index=daily_ret.index, dtype=float)
    current_w = w.copy()                      # 起始权重 = 目标

    for date, r in daily_ret.iterrows():
        port_ret[date] = (current_w * r).sum()       # 今日收益
        new_w = current_w * (1 + r)                  # 权重漂移
        current_w = new_w / new_w.sum()              # 归一化
        if date in rebalance_dates:                  # 调仓日收盘后
            current_w = w.copy()                     # 拉回目标

    return port_ret

In [ ]:
port_ret = monthly_rebalance_portfolio(
    all_df,
    weights=dict([(x[1],x[2]) for x in codes]),
    freq='M'
)
# 组合净值（初始 1）
nav = (1 + port_ret).cumprod()
print("年化收益:", ep.annual_return(port_ret))
print("年化波动:", ep.annual_volatility(port_ret))
print("Sharpe:", ep.sharpe_ratio(port_ret))
print("Sortino:", ep.sortino_ratio(port_ret))
print("最大回撤:", ep.max_drawdown(port_ret))
nav.plot(figsize=(10,5))